In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens
from src.fewshot.coref_helper import example_to_string, format_profile_for_prompt
from src.ann_extractor import extract_parent_level_annotations
from src.htmlLabel import ReferenceMention
from datetime import datetime
from src.tokenizer_utils import tokenize, decode
from src.models import get_messages
from tqdm import tqdm
import random

from src.main_coref import extract_docid_from_generation, dict_to_clusters



c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose the prompting configuration

In [2]:
filename = "2021QCCA1675"
split = "dev"
filepath = Path(DATA_DIR) / "annotated" / split / f"{filename}.html"

with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

gold_mentions = extract_parent_level_annotations(html_content)


#### Common Few SHot Selection

In [24]:
fewshot_method = "random"   # "greedy" | "random"

fewshot_filename = f"examples_coref_{fewshot_method}"

with open(FEWSHOT_CACHE_DIR / f"{fewshot_filename}.json", "r", encoding="utf-8") as f:
    fewshot_file_content = json.load(f)

fewshot_examples = [
    (
        json.loads(example["input"]),
        example["output"],
        example["meta"],
    )
    for example in fewshot_file_content["examples"]
]
print("fewshot examples from :", fewshot_filename)


fewshot examples from : examples_coref_random


##### Few Shot processing step

In [4]:
from src.rpr import ReferenceProfileRegistry
import random 

def sample_reference_profile_subset(
    rpr: ReferenceProfileRegistry,
    docid,
    doc_type: str = None,
    length: int = None,
    min_length: int = None,
    max_length: int = None,
    seed: int = None,
) -> ReferenceProfileRegistry:
    """
    Build a random subset of `rpr` as a new ReferenceProfileRegistry.

    Parameters
    ----------
    rpr : ReferenceProfileRegistry
        The full list of profiles to sample from.
    docid :
        The docid we care about when deciding inclusion.
    doc_type : str, optional
        If given, only profiles with this `doc_type` are considered for the subset.
    include_docid : {"yes", "no", "random"}
        - "yes":    the profile with `docid` is forced into the subset.
        - "no":     the profile with `docid` is forced OUT of the subset.
        - "random": the profile with `docid` is included with probability `p`.
    length : int, optional
        Exact size of the returned subset. If given, takes priority over
        min_length/max_length.
    min_length, max_length : int, optional
        If `length` is not given, the subset size is drawn uniformly from
        [min_length, max_length] (inclusive), using `seed`.
    seed : int, optional
        Seed for all the random choices made in this function (size choice,
        whether to include the target docid in "random" mode, and which
        other profiles fill the rest of the subset). Uses a local
        random.Random instance, so global random state is untouched.
    p : float, default 0.7
        Probability of including the target docid's profile when
        include_docid="random". Ignored otherwise.

    Returns
    -------
    ReferenceProfileRegistry
        A new list containing the sampled subset of profiles.
    """

    rng = random.Random(seed)

    all_profiles = list(rpr)
    if doc_type is not None:
        all_profiles = [prof for prof in all_profiles if prof.doc_type == doc_type]

    target_profile = rpr.get_profile_by_docid(docid)


    # Pool of profiles eligible to fill the "free" slots of the subset
    # (everything except the target profile, which is handled separately).
    other_profiles = [prof for prof in all_profiles if prof is not target_profile]


    if length is not None:
        subset_size = length
    else:
        if min_length is None or max_length is None:
            raise ValueError(
                "Either `length`, or both `min_length` and `max_length`, must be provided"
            )
        if min_length > max_length:
            raise ValueError("min_length cannot be greater than max_length")
        subset_size = rng.randint(min_length, max_length)

    if subset_size < 0:
        raise ValueError("Computed subset size is negative")

    # How many additional (non-target) profiles do we need to fill the subset?
    remaining_slots = subset_size - 1 if target_profile else subset_size
    remaining_slots = max(remaining_slots, 0)

    chosen_others = rng.sample(other_profiles, min(remaining_slots, len(other_profiles))) if remaining_slots > 0 else []

    subset_profiles = list(chosen_others)
    if target_profile is not None:
        subset_profiles.append(target_profile)

    # Shuffle so the target profile (if forced in) isn't always last.
    rng.shuffle(subset_profiles)

    result = ReferenceProfileRegistry()
    for prof in subset_profiles:
        result.add_profile(prof)

    return result


def format_profile_for_prompt(profile_dict: dict, max_fragments: int = 10) -> dict:
    """
    Take one profile's to_dict() output (with tracked fields still in
    {value: first_seen_id} form) and turn it into a prompt-friendly dict:
    - tracked fields become plain lists of their keys (ids dropped)
    - empty lists are omitted entirely
    - fragments_mentioned is truncated to the last `max_fragments` items
    """
    tracked_fields = ("alternative_titles", "citations", "fragments_mentioned", "authors")

    formatted = {}
    for key, value in profile_dict.items():
        if key in tracked_fields:
            values_list = list(value.keys()) if isinstance(value, dict) else list(value)
            if key == "fragments_mentioned":
                values_list = values_list[-max_fragments:]
            if not values_list:
                continue  # drop empty lists
            formatted[key] = values_list
        else:
            formatted[key] = value

    return formatted



def example_to_string(example_input: dict, docid: str, doctype: str, max_fragments: int = 10) -> str:
    """
    Given one fewshot example's `input` dict (with keys "input_mention",
    "context", "profileRegistry"), build a single formatted string
    describing the input. Output mention is intentionally not included.
    """
    rpr = ReferenceProfileRegistry.from_dict(example_input["profileRegistry"])

    attributes = ["docid", "alternative_titles",
                  "citations", "fragments_mentioned", "authors"]

    
    filtered_rpr = sample_reference_profile_subset(
        rpr=rpr,
        docid=docid,
        doc_type=doctype,
        min_length = 1,
        max_length = 10,
        seed=None,
    )
    

    #rpr_main_title = filtered_rpr.replace_docid_with_main_title()
        
    profiles_formatted = [
        format_profile_for_prompt(profile.to_dict(attributes=attributes), max_fragments=max_fragments)
        for profile in filtered_rpr
    ]

    lines = []
    lines.append(f"Input mention: {example_input['input_mention']}")
    lines.append(f"Context: {example_input['context']}")
    lines.append("Reference Profile Registry:")
    for i, profile in enumerate(profiles_formatted):
        lines.append(f"  Profile {i}: {profile}")

    return "\n".join(lines)

In [26]:
seed = 24
nb_fewshot_examples = 8
from src.rpr import ReferenceProfileRegistry

rng = random.Random(seed)

final_fewshot = []
for i in range(nb_fewshot_examples):
    example = fewshot_examples[i]
    input_, output, meta = example

    final_input = example_to_string(input_, docid=meta["docid"], doctype=input_["input_mention"].split(">")[0][1:])
    final_fewshot.append((final_input, output))

    print(final_input)
    print("-------------------------")
    print("output:", output)
    print("-------------------------")

    print("\n")


Input mention: <decision><title>Alberta Teachers’ Association</title></decision>
Context: ); ATCO Gas; Mouvement laïque; Igloo Vikski; Edmonton East) and bedrock judgments affirming the relevance of administrative expertise to the standard of review analysis and to “home statute” deference (C.U.P.E.; National Corn Growers; Domtar Inc.; Bradco Construction; Southam; Pushpanathan; 
Reference Profile Registry:
  Profile 0: {'docid': 'Ranville', 'alternative_titles': ['Minister of Indian Affairs and Northern Development v. Ranville'], 'citations': ['[1982] 2 S.C.R. 518'], 'fragments_mentioned': ['p. 528', 'p. 527']}
  Profile 1: {'docid': 'Planned Parenthood', 'alternative_titles': ['Planned Parenthood of Southeastern Pennsylvania v. Casey, Governor of Pennsylvania', 'Casey'], 'citations': ['505 U.S. 833 (1992)'], 'fragments_mentioned': ['p. 866', 'p. 864']}
  Profile 2: {'docid': 'Alberta Teachers', 'alternative_titles': ['Alberta (Information and Privacy Commissioner) v. Alberta Teachers

#### Common Prompt loading

In [27]:
prompt_filename = "coref_long.txt"

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

system_prompt used :  coref_long.txt


#### Assistant loading

In [28]:
MODEL_MAPPING_NAME = {
    "qwen7b": "Qwen2.5-7B-Instruct",
    "qwen32b": "Qwen2.5-32B-Instruct",
    "gpt-5.2": "gpt-5.2",
    "phi-4": "phi-4",
    "saul-54b": "SaulLM-54B-Instruct"
}

from src.models import AssistantFactory

model = "gpt-5.2"

if model == "gpt-5.2":
        assistant = AssistantFactory.create_from_config({
            "type": "openai",
            "model_name": model,
            "temperature": 1,
        })
else:
    assistant = AssistantFactory.create(MODEL_MAPPING_NAME[model])


#### Main processing fonction

In [29]:
rpr = ReferenceProfileRegistry()

attributes = ["docid", "alternative_titles",
                  "citations", "fragments_mentioned", "authors"]

config = LabelTransformConfig(
    use_simplified=True,
    switch_type=False,
    keep_attributes=["labelname"]
)  # We only remove the attribute

max_fragments = 10

mentions = extract_parent_level_annotations(html_content)
result = {}

failed_count = 0
log_path = "processing_log_1.txt"

with open(log_path, "a", encoding="utf-8") as log_file:
    
    for idx, mention in tqdm(enumerate(mentions), total=len(mentions)):
        id = mention.html_tag.attributes["id"]
        prepared_tokens = prepare_label_tokens(tokenize(mention.html_str), config)

        profiles_formatted = [
                format_profile_for_prompt(profile.to_dict(attributes=attributes), max_fragments=max_fragments)
                for profile in rpr
            ]

        lines = []
        lines.append("Reference Profile Registry:")
        for i, profile in enumerate(profiles_formatted):
            lines.append(f"  Profile {i}: {profile}")

        profiles_str ="\n".join(lines)


        user_input = decode(prepared_tokens) + "\n" + profiles_str

        messages = get_messages(
            system_prompt=system_prompt,
            user_input=user_input,
            fewshot_examples=final_fewshot,
            has_system_role=assistant.has_system_role,
            prefix="Annotate this mention: ",
        )

        generated = assistant.generate(messages=messages)

        docid_generated = extract_docid_from_generation(generated)
        if docid_generated is None:
            failed_count += 1
            print(f"Failed to extract docid for mention {idx} (id={id})")
            continue
        #print(f"generated : {generated} | docid extracted : {docid_generated}")

        

        mention.html_tag.set_attribute("docid", docid_generated)

        profile = rpr.update_from_mention(ReferenceMention(mention.html_str))
            

        result[id] = docid_generated



                # ---- logging ----
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "id": id,
            "user_input": decode(prepared_tokens),
            "profile_registry": profiles_str,
            "docid_generated": docid_generated,
            "profile_docid": profile.docid,
        }

        log_file.write(f"{'='*80}\n")
        log_file.write(f"MENTION #{idx}\n")
        log_file.write(f"{'='*80}\n")
        log_file.write(json.dumps(log_entry, indent=2, default=str, ensure_ascii=False))
        log_file.write("\n\n")
        log_file.flush()


print(f"Failed: {failed_count} / {len(mentions)}")

100%|██████████| 99/99 [02:45<00:00,  1.67s/it]

Failed: 0 / 99


In [30]:
for key, value in result.items():
    print(f"{key}: {value}")

1: Mouvement laïque québécois v. Saguenay (City)
4: Code de procédure pénale (Québec)
6: Act respecting the laicity of the State
10: Act respecting the laicity of the State
12: Canadian Charter of Rights and Freedoms
15: Act respecting the laicity of the State
17: Code de procédure pénale (Québec)
20: Code de procédure pénale (Québec)
23: Code de procédure pénale (Québec)
28: Code de procédure pénale (Québec)
31: Harper v. Canada (Attorney General)
33: Canadian Charter of Rights and Freedoms
35: Act respecting the laicity of the State
37: Mosaic Potash Esterhazy Limited Partnership v. Saskatchewan (Environment)
40: Mosaic Potash Esterhazy Limited Partnership v. Saskatchewan (Environment)
42: Canada (Attorney General) v. Oshkosh Defense Canada Inc.
46: Canadian Charter of Rights and Freedoms
48: Canadian Charter of Rights and Freedoms
51: Canadian Charter of Rights and Freedoms
53: Canadian Charter of Rights and Freedoms
55: Canadian Charter of Rights and Freedoms
57: Canadian Charter o

#### Capacity-bounded incremental entity linking with deferred multi-pass resolution

In [18]:
"""
Capacity-bounded incremental entity linking with deferred multi-pass resolution.

Extends the LLMLINK-style single-pass loop (Zhu et al., COLING 2025) with a
bounded ReferenceProfileRegistry, inspired by the bounded-memory idea in
Toshniwal et al. 2020 ("Learning to Ignore: Long Document Coreference with
Bounded Memory Neural Networks") and later work such as MEIC-DT (Luo et al.
2025). Unlike those approaches (which *evict* entities from memory when full),
this variant *defers* mentions to a subsequent pass with a freshly reset
registry, so no information is discarded -- only postponed.

Key invariant, per pass:
  - The registry never holds more than `max_registry_size` profiles.
  - A mention that matches an EXISTING profile in the current registry is
    always resolved immediately, regardless of registry fullness.
  - A mention that the LLM judges to be a NEW profile is only accepted if
    there is still room in the registry. Otherwise it is deferred to the
    next pass (registry reset, mentions processed again in original order).
  - A pass that defers 100% of its input mentions makes no progress -> stop
    and mark the remainder as unresolved (safety net against infinite loops).

Setting max_registry_size = 1 degenerates to a pure binary task: "does this
mention belong to the single currently-open profile, or not?".
"""

import json
from datetime import datetime

from tqdm import tqdm



def resolve_with_bounded_registry(
    mentions,
    max_registry_size,
    *,
    prepare_label_tokens,
    tokenize,
    config,
    format_profile_for_prompt,
    decode,
    get_messages,
    system_prompt,
    final_fewshot,
    assistant,
    extract_docid_from_generation,
    ReferenceMention,
    attributes=None,
    max_fragments=10,
    log_path="processing_log_bounded.txt",
    max_passes=None,
):
    """
    Run capacity-bounded, multi-pass incremental entity linking over `mentions`.

    Parameters mirror the pieces already used in your single-pass script; they
    are passed in explicitly (rather than imported globally) so this function
    stays reusable/testable outside your notebook/script context.

    Returns
    -------
    result : dict[str, str | None]
        mention_id -> resolved docid (None for mentions that could never be
        resolved, either because generation failed or because no progress
        could be made in a stalled pass).
    stats : dict
        Bookkeeping: number of passes, per-pass counts, total failures.
    """
    if attributes is None:
        attributes = [
            "main_title", "alternative_titles",
            "citations", "fragments_mentioned", "authors",
        ]

    result = {}
    failed_count = 0
    remaining_mentions = list(mentions)
    pass_num = 0
    pass_stats = []

    with open(log_path, "a", encoding="utf-8") as log_file:

        while remaining_mentions:
            pass_num += 1
            if max_passes is not None and pass_num > max_passes:
                # Hard safety cap: whatever's left is marked unresolved.
                for mention in remaining_mentions:
                    mid = mention.html_tag.attributes["id"]
                    result.setdefault(mid, None)
                    failed_count += 1
                break

            rpr = ReferenceProfileRegistry()
            deferred = []
            resolved_this_pass = 0

            for mention in tqdm(remaining_mentions, desc=f"Pass {pass_num}"):
                mid = mention.html_tag.attributes["id"]
                prepared_tokens = prepare_label_tokens(tokenize(mention.html_str), config)

                profiles_formatted = [
                    format_profile_for_prompt(
                        profile.to_dict(attributes=attributes),
                        max_fragments=max_fragments,
                    )
                    for profile in rpr
                ]
                lines = ["Reference Profile Registry:"]
                for i, profile_str in enumerate(profiles_formatted):
                    lines.append(f"  Profile {i}: {profile_str}")
                profiles_str = "\n".join(lines)

                user_input = decode(prepared_tokens) + "\n" + profiles_str

                messages = get_messages(
                    system_prompt=system_prompt,
                    user_input=user_input,
                    fewshot_examples=final_fewshot,
                    has_system_role=assistant.has_system_role,
                    prefix="Annotate this mention: ",
                )

                generated = assistant.generate(messages=messages)
                docid_generated = extract_docid_from_generation(generated)

                if docid_generated is None:
                    failed_count += 1
                    print(f"[pass {pass_num}] Failed to extract docid for mention (id={mid})")
                    result.setdefault(mid, None)
                    continue

                # --- capacity check happens BEFORE we let update_from_mention
                # possibly create a brand new profile ---
                existing_profile = rpr.get_profile_by_docid(docid_generated)
                registry_full = len(rpr.profiles) >= max_registry_size

                if existing_profile is None and registry_full:
                    # LLM wants a new entity, but there's no room left this
                    # pass -> defer, try again next pass with a clean slate.
                    deferred.append(mention)
                    continue

                mention.html_tag.set_attribute("docid", docid_generated)
                profile = rpr.update_from_mention(ReferenceMention(mention.html_str))
                if profile is None:
                    # parse_full_html failed downstream; treat like a failure.
                    failed_count += 1
                    result.setdefault(mid, None)
                    continue

                result[mid] = docid_generated
                resolved_this_pass += 1

                log_entry = {
                    "timestamp": datetime.now().isoformat(),
                    "pass": pass_num,
                    "id": mid,
                    "user_input": decode(prepared_tokens),
                    "docid_generated": docid_generated,
                    "main_title": profile.main_title,
                    "profile_docid": profile.docid,
                    "registry_size_after": len(rpr.profiles),
                }
                log_file.write(f"{'='*80}\n")
                log_file.write(f"PASS {pass_num} | MENTION id={mid}\n")
                log_file.write(f"{'='*80}\n")
                log_file.write(json.dumps(log_entry, indent=2, default=str, ensure_ascii=False))
                log_file.write("\n\n")
                log_file.flush()

            pass_stats.append({
                "pass": pass_num,
                "input": len(remaining_mentions),
                "resolved": resolved_this_pass,
                "deferred": len(deferred),
            })

            if resolved_this_pass == 0 and deferred:
                # No progress at all this pass -> further passes would loop
                # forever. Mark the rest unresolved and stop.
                print(
                    f"[pass {pass_num}] No progress made "
                    f"({len(deferred)} mentions still deferred) -> stopping."
                )
                for mention in deferred:
                    mid = mention.html_tag.attributes["id"]
                    result.setdefault(mid, None)
                    failed_count += 1
                break

            remaining_mentions = deferred

    stats = {
        "n_passes": pass_num,
        "pass_stats": pass_stats,
        "failed_count": failed_count,
        "total_mentions": len(mentions),
    }
    print(f"Done in {pass_num} pass(es). Failed: {failed_count} / {len(mentions)}")
    return result, stats

In [19]:

config = LabelTransformConfig(
    use_simplified=True,
    switch_type=False,
    keep_attributes=["labelname"]
)  # We only remove the attribute

max_fragments = 10

mentions = extract_parent_level_annotations(html_content)

result, stats = resolve_with_bounded_registry(
    mentions=mentions,
    max_registry_size=20,          # votre K, 1 = tâche binaire
    prepare_label_tokens=prepare_label_tokens,
    tokenize=tokenize,
    config=config,
    format_profile_for_prompt=format_profile_for_prompt,
    decode=decode,
    get_messages=get_messages,
    system_prompt=system_prompt,
    final_fewshot=final_fewshot,
    assistant=assistant,
    extract_docid_from_generation=extract_docid_from_generation,
    ReferenceMention=ReferenceMention,
    max_fragments=max_fragments,
    log_path="processing_log_bounded.txt",
    max_passes=None,   # garde-fou, ajustez selon la taille du doc
)

Pass 3: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]

Done in 3 pass(es). Failed: 0 / 99


In [31]:
system_clusters = dict_to_clusters(result)
with open("../private/system_clusters_2021_5.json", "w", encoding="utf-8") as f:
    json.dump(system_clusters, f, indent=2, ensure_ascii=False)

In [13]:
ground_truth = {mention.html_tag.attributes["id"]: mention.html_tag.attributes.get("docid") for mention in gold_mentions}

In [14]:
cluster_ground_truth = dict_to_clusters(ground_truth)
with open("../private/ground_truth_clusters_2021.json", "w", encoding="utf-8") as f:
    json.dump(cluster_ground_truth, f, indent=2, ensure_ascii=False)

#### Evaluation

CaNLL shared tasks (2011-2012) standardized three metrics : 
MUC (Vilain et al. 1995) : Measures how many links must be added/deleted to transform one clustering into another.
B³ (Bagga & Baldwin)
CEAF : Finds the optimal one-to-one alignment between predicted and gold clusters.
LEA : Weights entities according to their importance.
The famous CoNLL score is simply

(MUC + B³ + CEAF)/3


In [32]:
from src.evaluation.evaluation_coref import evaluate_coref, print_evaluation_table
scores = evaluate_coref(cluster_ground_truth, system_clusters)

print_evaluation_table(scores)


=== Coreference Evaluation ===
┌──────────┬────────────┬────────────┬────────────┐
│ Metric   │ Precision  │ Recall     │ F1         │
├──────────┼────────────┼────────────┼────────────┤
│ MUC      │ 0.8906     │ 0.8507     │ 0.8702     │
│ B3       │ 0.9214     │ 0.8670     │ 0.8934     │
│ CEAF_e   │ 0.8067     │ 0.8823     │ 0.8428     │
│ LEA      │ 0.8182     │ 0.7744     │ 0.7957     │
└──────────┴────────────┴────────────┴────────────┘
CoNLL average F1 (MUC/B3/CEAF_e): 0.8688



'\n=== Coreference Evaluation ===\n┌──────────┬────────────┬────────────┬────────────┐\n│ Metric   │ Precision  │ Recall     │ F1         │\n├──────────┼────────────┼────────────┼────────────┤\n│ MUC      │ 0.8906     │ 0.8507     │ 0.8702     │\n│ B3       │ 0.9214     │ 0.8670     │ 0.8934     │\n│ CEAF_e   │ 0.8067     │ 0.8823     │ 0.8428     │\n│ LEA      │ 0.8182     │ 0.7744     │ 0.7957     │\n└──────────┴────────────┴────────────┴────────────┘\nCoNLL average F1 (MUC/B3/CEAF_e): 0.8688\n'